# 01 - Data audit  (Stage 0)

**Purpose: lock D1 (news source) and D4 (sample window).** Both candidates are downloaded and
audited side by side. No description of either dataset is trusted - including the plan's - so every
claim is checked against the data here.

Criteria, in the plan's priority order:

1. **usable intraday timestamps with a known timezone** - a *gate*, not a score
2. coverage / duplication quality
3. sample length

**Deadline:** if neither candidate passes criterion 1 by the end of day 1, invoke **D2** (take the
better date-only set, map news dated *d* to trading day *d+1*, drop RQ2) and move on. Do not spend
day 2 hunting for better data.

This notebook contains no analysis logic. It imports from `src/`, calls, and displays.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
pd.set_option('display.width', 140); pd.set_option('display.max_columns', 40)

import config
from src import data, audit, plots


## 1. Load both candidates

`load_news` applies each source's documented timezone explicitly - it never infers one from the
data. If a path below does not exist, run `python data/raw/download.py --all` from the repo root.


In [ ]:
CANDIDATES = {
    'fnspid':   (config.DATA_RAW / 'fnspid.parquet',  'fnspid'),
    'benzinga': (config.DATA_RAW / 'benzinga.csv',    'benzinga'),
}

raw = {}
for name, (path, source) in CANDIDATES.items():
    if not path.exists():
        print(f'{name:9s} MISSING at {path}')
        continue
    raw[name] = data.load_news(path, source=source)
    lo, hi = raw[name]['ts_et'].min(), raw[name]['ts_et'].max()
    print(f'{name:9s} {len(raw[name]):>10,} headlines   {lo:%Y-%m-%d} .. {hi:%Y-%m-%d}')


## 2. Criterion 1 - are the timestamps real?

The tell is concentration. A date-only dump parsed to datetimes puts every row at the same clock
time, usually midnight. `timestamp_profile` returns a verdict, not a number to squint at.

**Record each dataset's source timezone from its documentation here, in writing, before reading the
plot** - the profile only proves the stamps vary, not that they were localized correctly.


In [ ]:
profiles = {name: audit.timestamp_profile(df) for name, df in raw.items()}
for name, p in profiles.items():
    print(f'{name}: ' + p['verdict'])

pd.DataFrame(profiles).T[['distinct_minutes_of_day', 'modal_time', 'modal_share',
                          'midnight_share', 'share_before_open', 'share_in_session',
                          'share_after_close', 'looks_date_only']]


In [ ]:
fig = plots.figure_timestamp_audit(
    {name: audit.hour_histogram(df) for name, df in raw.items()},
    {name: p['verdict'] for name, p in profiles.items()},
)
plots.save(fig, 'figure0_timestamp_audit'); fig


## 3. Criteria 2 and 3 - coverage, duplication, sample length

`zero_news_share` is a sample-size fact, not a footnote: those sessions are dropped from every
regression under D9. `thin_days_share` counts sessions that keep S_t but lose d_t (n_t < 5).


In [ ]:
calendar = data.trading_calendar('2010-01-01', '2025-12-31')
audits = {name: audit.audit_candidate(df, name, calendar) for name, df in raw.items()}

for name, a in audits.items():
    c, d = a['coverage'], a['duplication']
    print(f'\n=== {name} ===')
    print(f"  headlines      {c['n_headlines']:>10,}   dedup rate {d['dedup_rate']:.1%}"
          f" (exact {d['n_exact_dropped']:,} / near {d['n_near_dropped']:,})")
    print(f"  per session    mean {c['per_day_mean']:.1f}  median {c['per_day_median']:.0f}"
          f"  p10 {c['per_day_p10']:.0f}")
    print(f"  zero-news      {c['zero_news_days']:,} sessions ({c['zero_news_share']:.1%})")
    print(f"  thin (n<5)     {c['thin_days_share']:.1%} of sessions lose d_t")
    print(f"  unassignable   {c['n_unassignable']:,} headlines fall outside the calendar")


In [ ]:
for name, a in audits.items():
    print(f'\n{name} headlines per year')
    print(a['coverage']['per_year'].to_string())


### Ticker-tag sanity, 50 rows

Read these. Tags are used only in the Stage 6 single-name spot check, so this is a sanity check,
not a validation.


In [ ]:
for name, df in raw.items():
    s = audit.ticker_sanity(df, n=50)
    print(f"\n=== {name} ===  untagged {s.attrs['untagged_share']:.1%}, "
          f"mean tags {s.attrs['mean_tags']:.2f}")
    display(s.head(15))
    display(s.attrs['most_covered'])


## 4. The D1 decision

Criterion 1 is a **gate**: a date-only dataset is disqualified no matter how good its coverage is.
Among candidates that clear it, the ranking is fewer duplicates -> fewer zero-news days -> longer
window. If every candidate fails the gate, the recommendation is D2.


In [ ]:
choice = audit.compare_candidates(audits)
display(choice)
print('\nRECOMMENDATION:', choice.attrs['recommendation'])
print('REASON:', choice.attrs['reason'])


## 5. The D4 decision

`suggest_window` proposes the longest *recent* run of years with stable coverage. Stability, not
length, is binding: a year with a tenth of the neighbouring coverage makes S_t a different
measurement. It is a proposal - read `per_year` above and lock D4 by hand.


In [ ]:
w = audits[choice.attrs['recommendation']]['window']
print(w['reason'])
print('excluded years:', w['excluded_years'])
print(f"clears the {config.MIN_TRADING_DAYS:,} trading-day target: {w['ok']}")


---
## 6. Write the decision into `config.py`, then stop

Fill in the two PENDING cells and **freeze them**:

```python
NEWS_SOURCE   = '...'         # D1
NEWS_RAW_PATH = DATA_RAW / '...'
SAMPLE_START  = 'YYYY-MM-DD'  # D4
SAMPLE_END    = 'YYYY-MM-DD'
```

Changing either after seeing an Act-2 result is exactly the look-ahead this design exists to
prevent. If you must, log the change and the date in `future-work.md`.

Then build the market table and time the expensive scorer before committing to Stage 1:

```bash
python data/raw/download.py --market   # -> interim/market.parquet
python rescore.py --time-only          # extrapolate the FinBERT pass first
```


In [ ]:
# Build interim/headlines.parquet for the locked source and window.
# Runs only once config.NEWS_SOURCE and the D4 dates are filled in.
assert config.NEWS_SOURCE and config.SAMPLE_START, 'lock D1 and D4 in config.py first'

clean, stats = data.dedup(raw[config.NEWS_SOURCE])
print(f"dedup: {stats['n_in']:,} -> {stats['n_out']:,}  ({stats['dedup_rate']:.1%} removed)")

in_window = clean['ts_et'].between(config.SAMPLE_START, config.SAMPLE_END + ' 23:59:59')
headlines = clean[in_window].reset_index(drop=True)
config.HEADLINES_PARQUET.parent.mkdir(parents=True, exist_ok=True)
headlines.to_parquet(config.HEADLINES_PARQUET, index=False)
print(f'wrote {config.HEADLINES_PARQUET}  ({len(headlines):,} rows)')


---
## 7. Stage 3 extension - EDA on the finished panel

Come back here after `run_all.py` has built `daily_panel.parquet`.

> **Note that changes what you may claim later:** if `corr(s_finbert, s_lm) > 0.9` daily, the
> attenuation comparison (6.3) has little room to separate the scorers. Report it as bounded and
> inconclusive. Do not redesign the comparison to manufacture a difference.


In [ ]:
panel = pd.read_parquet(config.PANEL_PARQUET)
print(f'{len(panel):,} sessions; zero-news {int((panel.n_headlines == 0).sum()):,}')

display(panel[[f's_{s}' for s in config.SCORERS]].corr())   # the note above
display(plots.figure4_context(panel))                       # Figure 4
display(plots.figure_acf(panel))                            # justifies D10 and D12
display(plots.figure_coverage(panel))                       # coverage drift
